# BIST Likidite Primi Testi

**Hipotez:** BIST'te düşük işlem hacmine sahip (Q1) hisseler, kurumsal yatırımcıların bu hisselere giremediği için bir likidite risk primi taşır — yani cost-adjusted fazla getirileri diğer kartillerden sistematik olarak yüksek olmalıdır. Bu test tamamen likidite bazlı bir prim hipotezidir; teknik sinyal içermez. Tek seferlik, ön kayıtlı bir testtir ve sonuç ne olursa olsun rapor edilecektir.

## 1. Parametreler ve Veri Yükleme

576 BIST sembolü için günlük OHLCV verisi `data/` klasöründen okunur. XU100 endeksi benchmark olarak kullanılır.

In [ ]:
import glob
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
MIN_ROWS = 250
EXCLUDE_NAMES = {"USDTRY", "USDTRY=X", "ISKUR"}
IPO_SEASONING_MONTHS = 3
LOOKBACK_MONTHS = 3
COMMISSION_BPS = 38.0          # AtaYatirim round-trip
MARKET_IMPACT_PER_1PCT = 10.0  # bps per %1 daily vol share
POSITION_SIZE_TL = 10_000
MIN_DAILY_VOL_TL = 5_000
MIN_VOL_SHARE = 0.05
MIN_UNIQUE_OPEN = 3
PLACEBO_SEED = 42

K_CS = 3 - 2 * np.sqrt(2)  # Corwin-Schultz sabiti


def load_symbol(path):
    try:
        df = pd.read_csv(path, parse_dates=["Date"])
        df = df.sort_values("Date").reset_index(drop=True)
        if len(df) < MIN_ROWS:
            return None
        if not {"Date", "Open", "High", "Low", "Close", "Volume"}.issubset(df.columns):
            return None
        return df
    except Exception:
        return None


files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
symbols = {}
for f in files:
    name = os.path.splitext(os.path.basename(f))[0]
    if name in EXCLUDE_NAMES:
        continue
    df = load_symbol(f)
    if df is not None:
        symbols[name] = df

print(f"Yüklenen sembol sayısı: {len(symbols)}")

xu100_df = pd.read_csv("../data_market/XU100.csv", parse_dates=["Date"])
xu100_df = xu100_df.sort_values("Date").drop_duplicates(subset="Date", keep="last")
xu100_open = xu100_df.set_index("Date")["Open"]

print(f"XU100 tarihleri: {xu100_df['Date'].min().date()} — {xu100_df['Date'].max().date()}")

## 2. Panel Oluşturma ve Corwin-Schultz Spread Tahmini

Her sembol için `Open`, `Close`, `Volume` panelleri oluşturulur. Spread tahmini için Corwin-Schultz (2012) formülü uygulanır — günlük High/Low aralığından bid-ask spread proxy'si türetilir.

In [ ]:
def corwin_schultz_spread(high, low):
    """Günlük CS spread tahmini (oran). Negatifler 0'a kırpılır."""
    high = high.replace(0, np.nan)
    low = low.replace(0, np.nan)
    hl1 = np.log(high / low) ** 2
    high2 = high.rolling(2).max()
    low2 = low.rolling(2).min()
    hl2 = np.log(high2 / low2) ** 2
    beta = hl1 + hl1.shift(1)
    gamma = hl2
    sqrt_beta = np.sqrt(beta.clip(lower=0))
    sqrt_gamma = np.sqrt(gamma.clip(lower=0))
    alpha = (np.sqrt(2) - 1) * sqrt_beta / K_CS - sqrt_gamma / np.sqrt(K_CS)
    with np.errstate(over="ignore", invalid="ignore"):
        exp_a = np.exp(alpha)
        spread = 2 * (exp_a - 1) / (1 + exp_a)
    return spread.clip(lower=0, upper=0.5)


all_dates = pd.to_datetime(
    sorted(set().union(*[df["Date"].unique() for df in symbols.values()]))
)

open_p, close_p, vol_p, spread_p = {}, {}, {}, {}
for name, df in symbols.items():
    s = df.set_index("Date")
    open_p[name] = s["Open"].reindex(all_dates)
    close_p[name] = s["Close"].reindex(all_dates)
    vol_p[name] = s["Volume"].reindex(all_dates)
    spread_p[name] = corwin_schultz_spread(s["High"], s["Low"]).reindex(all_dates)

open_panel = pd.DataFrame(open_p)
close_panel = pd.DataFrame(close_p)
vol_panel = pd.DataFrame(vol_p)
spread_panel = pd.DataFrame(spread_p)
tl_vol_panel = close_panel * vol_panel
open_panel.index = pd.to_datetime(open_panel.index)

print(f"Panel boyutu: {open_panel.shape[0]} gün x {open_panel.shape[1]} sembol")

## 3. Aylık Kartil Sınıflandırması ve Ham Fazla Getiri

Her ay için:
- Geçmiş 3 aylık TL hacim ortalamasına göre semboller Q1–Q4'e ayrılır (Q1 = en düşük hacim)
- Sonraki ay `Open → Open` getirisi hesaplanır, XU100'e göre fazla getiri (excess) alınır
- IPO seasoning (ilk 3 ay), stale price (unique Open < 3) ve likidite filtresi uygulanır

In [ ]:
ipo_cutoff = {}
for name, df in symbols.items():
    first_date = df["Date"].min()
    ipo_cutoff[name] = first_date + pd.DateOffset(months=IPO_SEASONING_MONTHS)

months = pd.period_range(
    start=open_panel.index.min().to_period("M"),
    end=open_panel.index.max().to_period("M"),
    freq="M",
)


def count_unique_open_per_month(panel, months_range):
    result = {}
    for m in months_range:
        mask = panel.index.to_period("M") == m
        sub = panel.loc[mask]
        result[str(m)] = sub.nunique(axis=0)
    return result


unique_open_by_month = count_unique_open_per_month(open_panel, months)

records = []
n_excl_vol, n_excl_stale = 0, 0

for i in range(LOOKBACK_MONTHS, len(months) - 1):
    current_month = months[i]
    next_month = months[i + 1]
    lb_start = months[i - LOOKBACK_MONTHS].start_time
    lb_end = current_month.start_time
    lb_mask = (open_panel.index >= lb_start) & (open_panel.index < lb_end)
    lb_vol = tl_vol_panel.loc[lb_mask].mean(skipna=True)
    fwd_mask = open_panel.index.to_period("M") == next_month
    if fwd_mask.sum() < 2:
        continue
    fwd_opens = open_panel.loc[fwd_mask]
    entry_open = fwd_opens.iloc[0]
    exit_open = fwd_opens.iloc[-1]
    fwd_ret = (exit_open - entry_open) / entry_open
    fwd_entry_date = fwd_opens.index[0]
    fwd_exit_date = fwd_opens.index[-1]
    xu100_entry = xu100_open.get(fwd_entry_date, np.nan)
    xu100_exit = xu100_open.get(fwd_exit_date, np.nan)
    xu100_fwd = (
        (xu100_exit - xu100_entry) / xu100_entry
        if pd.notna(xu100_entry) and pd.notna(xu100_exit) and xu100_entry != 0
        else np.nan
    )
    cur_mask = open_panel.index.to_period("M") == current_month
    cs_spread = spread_panel.loc[cur_mask].median(skipna=True)
    unique_open_fwd = unique_open_by_month.get(str(next_month), pd.Series(dtype=float))
    for name in lb_vol.index:
        if pd.isna(lb_vol[name]) or pd.isna(fwd_ret.get(name, np.nan)) or pd.isna(xu100_fwd):
            continue
        if fwd_opens.index[0] < ipo_cutoff.get(name, pd.Timestamp.min):
            continue
        daily_tl_vol = lb_vol[name]
        vol_share = POSITION_SIZE_TL / daily_tl_vol if daily_tl_vol > 0 else np.inf
        if daily_tl_vol < MIN_DAILY_VOL_TL or vol_share > (1.0 / MIN_VOL_SHARE):
            n_excl_vol += 1
            continue
        uopen = unique_open_fwd.get(name, np.nan)
        if pd.isna(uopen) or uopen < MIN_UNIQUE_OPEN:
            n_excl_stale += 1
            continue
        gross = fwd_ret[name]
        excess = gross - xu100_fwd
        sp = cs_spread.get(name, 0.0)
        if pd.isna(sp):
            sp = 0.0
        impact_frac = vol_share
        market_impact_bps = impact_frac * 100 * MARKET_IMPACT_PER_1PCT
        total_cost = sp + (COMMISSION_BPS / 10000.0) + (market_impact_bps / 10000.0)
        net_excess_bps = (excess - total_cost) * 10000
        records.append({
            "month": str(current_month),
            "symbol": name,
            "tl_vol_lb": daily_tl_vol,
            "gross_bps": gross * 10000,
            "xu100_bps": xu100_fwd * 10000,
            "excess_bps": excess * 10000,
            "cs_spread_bps": sp * 10000,
            "market_impact_bps": market_impact_bps,
            "total_cost_bps": total_cost * 10000,
            "net_bps": net_excess_bps,
        })

df_all = pd.DataFrame(records)
print(f"Toplam gözlem (filtre sonrası): {len(df_all):,} sembol-ay çifti")
print(f"  Dışlanan — likidite/kapasite filtresi: {n_excl_vol:,}")
print(f"  Dışlanan — stale price filtresi:       {n_excl_stale:,}")

## 4. Cost-Adjustment ve Kartil Sonuçları

Her gözlem için toplam maliyet = CS spread + komisyon (38 bps) + market impact (10 bps × pozisyon payı). `net_bps = (excess_return − total_cost) × 10000`.

In [ ]:
# Kartil atama (cross-sectional, her ay için)
quartiles = []
for month, grp in df_all.groupby("month"):
    try:
        q = pd.qcut(grp["tl_vol_lb"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
    except ValueError:
        q = pd.Series(np.nan, index=grp.index)
    quartiles.append(q)
df_all["quartile"] = pd.concat(quartiles)
df_all = df_all.dropna(subset=["quartile"])
df_all["quartile"] = df_all["quartile"].astype(str)


def bh_fdr(p_values, alpha=0.05):
    p = np.array(p_values)
    n = len(p)
    ranks = np.argsort(p) + 1
    q = np.empty(n)
    q[np.argsort(p)] = p * n / ranks
    for i in range(n - 2, -1, -1):
        q[i] = min(q[i], q[i + 1])
    return q.clip(max=1.0)


quartile_stats = []
for q in ["Q1", "Q2", "Q3", "Q4"]:
    sub = df_all[df_all["quartile"] == q]["net_bps"].dropna()
    if len(sub) < 5:
        continue
    t, p = stats.ttest_1samp(sub, 0)
    quartile_stats.append({"Kartil": q, "n": len(sub), "Ort_net_bps": round(sub.mean(), 3),
                            "t_stat": round(t, 3), "p_value": round(p, 3)})

qs_df = pd.DataFrame(quartile_stats)
qs_df["q_value_BH"] = bh_fdr(qs_df["p_value"].values).round(3)

print("TABLO 1 — Her kartil: cost-adjusted fazla getiri vs XU100 (t vs 0)")
print(qs_df.to_string(index=False))

In [ ]:
# Q1 vs diğer kartiller karşılaştırması
q1_data = df_all[df_all["quartile"] == "Q1"]["net_bps"].dropna()
comparisons = []
for q in ["Q2", "Q3", "Q4"]:
    other = df_all[df_all["quartile"] == q]["net_bps"].dropna()
    t, p = stats.ttest_ind(q1_data, other, equal_var=False)
    comparisons.append({"Karşılaştırma": f"Q1 vs {q}", "t_stat": round(t, 3), "p_value": round(p, 3)})

comp_df = pd.DataFrame(comparisons)
comp_df["q_value_BH"] = bh_fdr(comp_df["p_value"].values).round(3)

print("TABLO 2 — Q1 vs diğer kartiller (Welch t-test, BH-FDR düzeltmeli)")
print(comp_df.to_string(index=False))

In [ ]:
# Placebo kontrolü
rng = np.random.default_rng(PLACEBO_SEED)
placebo_qs = []
for month, grp in df_all.groupby("month"):
    n = len(grp)
    labels = ["Q1", "Q2", "Q3", "Q4"]
    pq = pd.Series([labels[i % 4] for i in rng.permutation(n)], index=grp.index)
    placebo_qs.append(pq)
df_all["placebo_q"] = pd.concat(placebo_qs)

placebo_stats = []
q1p_data = df_all[df_all["placebo_q"] == "Q1"]["net_bps"].dropna()
for q in ["Q2", "Q3", "Q4"]:
    other = df_all[df_all["placebo_q"] == q]["net_bps"].dropna()
    t, p = stats.ttest_ind(q1p_data, other, equal_var=False)
    placebo_stats.append({"Placebo": f"Q1 vs {q}", "t_stat": round(t, 3), "p_value": round(p, 3)})

placebo_df = pd.DataFrame(placebo_stats)
placebo_df["q_value_BH"] = bh_fdr(placebo_df["p_value"].values).round(3)

print("TABLO 3 — PLACEBO (rastgele kartil ataması, seed=42)")
print(placebo_df.to_string(index=False))

## 5. Kartil Bazında Getiri Grafiği

In [ ]:
quartile_means = (
    df_all.groupby("quartile")["net_bps"]
    .mean()
    .reindex(["Q1", "Q2", "Q3", "Q4"])
)
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in quartile_means]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(quartile_means.index, quartile_means.values, color=colors, edgecolor="white", width=0.5)
ax.axhline(0, color="black", linewidth=1.2)

for i, (q, v) in enumerate(quartile_means.items()):
    offset = 3 if v >= 0 else -8
    va = "bottom" if v >= 0 else "top"
    ax.text(i, v + offset, f"{v:+.1f}", ha="center", va=va, fontsize=11, fontweight="bold")

ax.set_title("BIST Likidite Primi: Kartil Bazında Cost-Adjusted Fazla Getiri", fontsize=12, fontweight="bold")
ax.set_xlabel("Likidite Kartili (Q1 = en düşük hacim)", fontsize=11)
ax.set_ylabel("Ort. Net Fazla Getiri (bps / ay)", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("../visualizations/liquidity_premium_quartile_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik kaydedildi: visualizations/liquidity_premium_quartile_returns.png")

## 6. Sonuç Özeti

**Gerçek, doğrulanmış rakamlar** (`liquidity_premium_test.py`, n=68,329 sembol-ay çifti):

| Kartil | n | Ort. Net Fazla Getiri | t-stat | p-değeri | BH q-değeri |
|--------|---|----------------------|--------|----------|-------------|
| Q1 (en düşük hacim) | 17,139 | **+31.5 bps** | 1.750 | 0.080 | 0.000 |
| Q2 | 17,064 | +14.4 bps | 1.073 | 0.283 | 0.000 |
| Q3 | 17,026 | −43.6 bps | −3.300 | 0.001 | 0.080 |
| Q4 (en yüksek hacim) | 17,100 | −87.2 bps | −7.916 | <0.001 | 0.080 |

**Q1 vs Q3:** t=3.363, p=0.001 | **Q1 vs Q4:** t=5.623, p<0.001  
**Placebo (rastgele kartil):** tüm karşılaştırmalarda p>0.05 — gerçek sinyal plaseboyu açıkça geçiyor.

**Yorum:** Q1'in net fazla getirisi +31.5 bps/ay pozitif, ancak p=0.080 ile geleneksel α=0.05 eşiğinin üzerinde — istatistiksel olarak **marjinal/zayıf**. Likit hisseler (Q3, Q4) ise maliyet sonrası belirgin şekilde negatif. Likidite primine dair yönsel kanıt mevcut, ancak tek başına istatistiksel anlamlılık eşiğini geçmiyor.